# UDT108 Project: Milk Adulteration Detection Using NIR Spectroscopy\n**Student:** Chetna Ahuja | **Roll No:** 2303210015 | **Course:** UDT108 - Project and Case Study\n**Dataset DOI:** 10.17632/3gxjgxkg76.1 (Mendeley Data NIR Spectral Repository)

In [1]:
import numpy as np\nimport pandas as pd\nfrom scipy.signal import savgol_filter\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler, LabelEncoder\n\n# 1. Load Mendeley NIR Dataset (900 nm to 1700 nm absorbance columns)\ndf_nir = pd.read_csv('Mendeley_NIR_Milk_Adulteration_900_1700nm.csv')\nwavelength_cols = [c for c in df_nir.columns if c not in ['Sample_ID', 'Adulterant_Class', 'Conc_wv']]\n\n# 2. Standard Normal Variate (SNV) Scatter Correction Function\ndef apply_snv(X_matrix):\n    mean_row = np.mean(X_matrix, axis=1, keepdims=True)\n    std_row = np.std(X_matrix, axis=1, keepdims=True)\n    return (X_matrix - mean_row) / (std_row + 1e-8)\n\n# 3. Chemometric Preprocessing Pipeline: SNV + Savitzky-Golay 1st Derivative (w=11, p=2)\ndef preprocess_nir(df_wavelengths):\n    X_snv = apply_snv(df_wavelengths.values)\n    X_sg1 = savgol_filter(X_snv, window_length=11, polyorder=2, deriv=1, axis=1)\n    return X_sg1\n\nX_proc = preprocess_nir(df_nir[wavelength_cols])\ny_vec = LabelEncoder().fit_transform(df_nir['Adulterant_Class'])\n\n# 4. Leakage-Free Stratified 80:20 Train-Test Split & Feature Standardization\nX_train, X_test, y_train, y_test = train_test_split(\n    X_proc, y_vec, test_size=0.20, random_state=42, stratify=y_vec\n)\nscaler = StandardScaler()\nX_train_sc = scaler.fit_transform(X_train)\nX_test_sc = scaler.transform(X_test)\n\nprint(f'[SUCCESS] Dataset Shape: {df_nir.shape} | Spectral Channels: {len(wavelength_cols)} (900.0 - 1700.0 nm)')\nprint(f'[SUCCESS] X_train_sc: {X_train_sc.shape}, X_test_sc: {X_test_sc.shape}')

In [2]:
from sklearn.ensemble import RandomForestClassifier\nfrom sklearn.svm import SVC\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import StratifiedKFold, cross_val_score\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score\n\nclassifiers = {\n    "Random Forest (n=300, Gini)": RandomForestClassifier(n_estimators=300, max_depth=16, criterion='gini', random_state=42, n_jobs=-1),\n    "Support Vector Machine (RBF)": SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=42),\n    "K-Nearest Neighbors (k=5)": KNeighborsClassifier(n_neighbors=5, weights='distance', metric='minkowski', p=2),\n    "Logistic Regression (L2)": LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)\n}\n\ncv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\nresults = []\n\nfor name, clf in classifiers.items():\n    cv_acc = cross_val_score(clf, X_train_sc, y_train, cv=cv5, scoring='accuracy')\n    clf.fit(X_train_sc, y_train)\n    y_pred = clf.predict(X_test_sc)\n    y_prob = clf.predict_proba(X_test_sc)\n    \n    acc = accuracy_score(y_test, y_pred) * 100\n    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')\n    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro') * 100\n    \n    results.append({\n        "Model": name,\n        "Test Accuracy (%)": round(acc, 2),\n        "5-Fold CV Mean (%)": f"{round(cv_acc.mean()*100, 2)} ± {round(cv_acc.std()*100, 2)}",\n        "Macro Precision (%)": round(p*100, 2),\n        "Macro Recall (%)": round(r*100, 2),\n        "Macro F1-Score (%)": round(f1*100, 2),\n        "ROC-AUC (%)": round(auc, 2)\n    })\n\ndf_res = pd.DataFrame(results)\nprint("=" * 110)\nprint("UDT108 PROJECT BENCHMARK: MILK ADULTERATION CLASSIFICATION (900-1700 nm)")\nprint("=" * 110)\nprint(df_res.to_string(index=False))\nprint("=" * 110)

In [3]:
from sklearn.metrics import classification_report, confusion_matrix\n\nrf_best = classifiers['Random Forest (n=300, Gini)']\nimportances = rf_best.feature_importances_\ntop_indices = np.argsort(importances)[::-1][:7]\n\nprint("--- TOP DIAGNOSTIC NIR WAVELENGTHS BY RANDOM FOREST GINI IMPORTANCE ---")\nfor rank, idx in enumerate(top_indices, 1):\n    print(f"Rank {rank}: Wavelength = {wavelength_cols[idx]} nm | Gini Importance = {importances[idx]:.4f}")\n\nprint("\n--- RANDOM FOREST PER-CLASS CLASSIFICATION AUDIT REPORT ---")\ny_pred_rf = rf_best.predict(X_test_sc)\nprint(classification_report(y_test, y_pred_rf, digits=3))\n\nprint("--- MULTI-CLASS CONFUSION MATRIX (RANDOM FOREST) ---")\nprint(confusion_matrix(y_test, y_pred_rf))